# 🧠 Brain Tumour Segmentation — Google Colab Training Notebook

This notebook trains the U-Net (ResNet-34 encoder) on the **LGG MRI
Segmentation** dataset, end to end, on a free Colab **GPU**.

**Before you start:** click `Runtime → Change runtime type → Hardware
accelerator → GPU (T4)`.

Each step has a plain-language explanation. Just run the cells top to bottom.


## Step 1 — Install the libraries

We install the same packages listed in `requirements.txt`. On Colab this takes
a minute or two. The `>=` versions pick whatever is compatible with Colab's
Python.


In [ ]:
!pip install -q "segmentation-models-pytorch>=0.3.3" \
    "albumentations>=1.3.1" "opencv-python>=4.8.0" "tifffile>=2023.9.26" \
    "scikit-learn>=1.3.0" "tqdm>=4.66.0" "PyYAML>=6.0"
print("Done installing.")


## Step 2 — Get the project code

Two options:

**A. Upload the project ZIP** (the one you were given) using the Files panel on
the left, then unzip it. Adjust the name if needed.

**B.** Or, if you have pushed it to GitHub, `git clone` it instead.

After this cell, your working directory should contain `train.py`, `evaluate.py`,
`src/`, `data/`, `config/`, etc.


In [ ]:
import os
# --- Option A: you uploaded brain_tumor_segmentation.zip ---
if os.path.exists("brain_tumor_segmentation.zip"):
    !unzip -q -o brain_tumor_segmentation.zip
    # If it unzipped into a sub-folder, step into it:
    if os.path.isdir("brain_tumor_segmentation"):
        %cd brain_tumor_segmentation

# --- Option B: git clone (uncomment and edit) ---
# !git clone https://github.com/<your-username>/brain_tumor_segmentation.git
# %cd brain_tumor_segmentation

print("Working directory:", os.getcwd())
!ls


## Step 3 — Download the dataset from Kaggle

The dataset is **not** bundled with the code (it is ~800 MB). You download it
straight onto the Colab machine with the Kaggle API.

**Get your `kaggle.json`:**
1. Go to kaggle.com → your profile → *Settings* → *API* → **Create New Token**.
2. That downloads `kaggle.json`.
3. Run the cell below and upload that file when prompted.


In [ ]:
from google.colab import files
print("Upload your kaggle.json (from Kaggle → Settings → API → Create New Token):")
uploaded = files.upload()   # choose kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle credentials installed.")


In [ ]:
# Download + unzip the LGG MRI Segmentation dataset (~800 MB).
!pip install -q kaggle
!kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
!unzip -q -o lgg-mri-segmentation.zip -d lgg-mri-segmentation
# The slices live under lgg-mri-segmentation/kaggle_3m/
!ls lgg-mri-segmentation/kaggle_3m | head


## Step 4 — Patient-level split

We split the 110 patients into train/val/test **by patient**, so no patient's
slices appear in two splits (this prevents the model from "cheating"). This
writes `data/split.csv`.


In [ ]:
!python data/prepare_data.py \
    --raw_root lgg-mri-segmentation/kaggle_3m \
    --out_csv data/split.csv


## Step 5 — (Optional) Verify the pipeline in seconds

This synthetic smoke test confirms every module connects, before you spend
~1.5 hours on real training. It needs no dataset.


In [ ]:
!python tests/test_pipeline.py


## Step 6 — Train the model 🚀

This trains the U-Net with the pretrained ResNet-34 encoder. On a free T4 GPU
expect **~1.5–2 hours** for the full 50 epochs (early stopping may end it
sooner). The best model (by validation Dice) is saved to
`results/best_model.pth` and the training curves to `results/curves.png`.

> Tip: to do a quick trial first, add `--epochs 5`.


In [ ]:
!python train.py            # add --epochs 5 for a fast trial


## Step 7 — Show the training curves inline

Loss should fall and Dice should rise. A healthy run plateaus with validation
Dice around 0.85–0.90.


In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("results/curves.png"):
    display(Image("results/curves.png"))
else:
    print("results/curves.png not found - did training finish?")


## Step 8 — Evaluate on the test set + compare to from-scratch

This reports Dice, IoU, sensitivity and specificity on the **held-out test
patients**, and trains a from-scratch U-Net (no ImageNet weights) to measure
how much the pretrained encoder helped — your transfer-learning contribution.

> The from-scratch baseline doubles the time. Use `--skip_scratch` to skip it,
> or `--scratch_epochs 15` for a faster baseline.


In [ ]:
!python evaluate.py --scratch_epochs 20


In [ ]:
# Show a few sample prediction overlays produced by evaluate.py
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("results/sample_prediction_*.png")):
    print(p)
    display(Image(p))


## Step 9 — Download the trained model

Save `best_model.pth` to your computer so the Streamlit app can use it locally.


In [ ]:
from google.colab import files
import os
if os.path.exists("results/best_model.pth"):
    files.download("results/best_model.pth")
else:
    print("results/best_model.pth not found - train first.")


---
### Done!
You now have a trained model, evaluation numbers, training curves and sample
overlays. Put `best_model.pth` into your local `results/` folder and run:

```bash
streamlit run app/streamlit_app.py
```
